# 3클래스 리텐션 상태 모델 v03

기존 Core 43개 피처와 시간 분할을 보존하고 다음 연도 상태를 유지·약화·중단으로 예측한다.
선정연도 2017은 모델·규제·임계값 선택에 사용하지 않고 최종 Test로 한 번만 평가한다.

In [1]:
from __future__ import annotations

import hashlib
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import yaml
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize

SEED = 42
CLASS_CODES = [0, 1, 2]
CLASS_NAMES = ['retained', 'weakened', 'stopped']
CLASS_LABELS_KO = {0: '파워 지위 유지', 1: '파워 지위 약화', 2: '리뷰 활동 중단'}
STOPPED_THRESHOLD = 0.45
WEAKENED_THRESHOLD = 0.36
PRIMARY_TARGET_RATE = 0.20
TOP_K_RATES = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
TIME_FOLDS = [(2011, 2012), (2012, 2013), (2013, 2014), (2014, 2015), (2015, 2016)]

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
MODEL_DIR = PROJECT_ROOT / 'models'
REPORT_TABLE_DIR = PROJECT_ROOT / 'reports' / 'tables'
REPORT_MODEL_DIR = PROJECT_ROOT / 'reports' / 'modeling'
PREDICTION_DIR = DATA_DIR / 'processed' / 'predictions'

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'analysis_config.yaml'
MODELING_PATH = DATA_DIR / 'processed' / 'modeling_dataset_rolling_v02.parquet'
COHORT_PATH = DATA_DIR / 'interim' / 'rolling' / 'culinary_rolling_cohort_master_v02.parquet'
REVIEW_PATHS = [
    DATA_DIR / 'interim' / 'restaurant_reviews.parquet',
    DATA_DIR / 'interim' / 'additional_culinary_reviews_v02.parquet',
]
V02_METADATA_PATH = MODEL_DIR / 'final_core_hgb_metadata_v02.json'

MODEL_PATH = MODEL_DIR / 'final_core_logistic_multiclass_v03.joblib'
METADATA_PATH = MODEL_DIR / 'final_core_logistic_multiclass_metadata_v03.json'
PROFILE_PATH = PREDICTION_DIR / 'final_test_retention_profiles_v03.parquet'
DISTRIBUTION_PATH = REPORT_TABLE_DIR / 'retention_state_distribution_v03.csv'
VALIDATION_PATH = REPORT_TABLE_DIR / 'multiclass_validation_results_v03.csv'
CONFUSION_PATH = REPORT_TABLE_DIR / 'multiclass_confusion_matrix_v03.csv'
TOP_K_PATH = REPORT_TABLE_DIR / 'multiclass_top_k_performance_v03.csv'
REPORT_PATH = REPORT_MODEL_DIR / 'multiclass_model_performance_v03.md'

for directory in [MODEL_DIR, REPORT_TABLE_DIR, REPORT_MODEL_DIR, PREDICTION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
state_config = config['retention_state']
assert state_config['retained_min_review_count'] == 10
assert state_config['retained_min_active_months'] == 3
assert state_config['stopped_review_count'] == 0

v02_metadata = json.loads(V02_METADATA_PATH.read_text(encoding='utf-8'))
feature_columns = v02_metadata['feature_columns']
assert len(feature_columns) == 43

modeling_df = pd.read_parquet(MODELING_PATH)
cohort_df = pd.read_parquet(COHORT_PATH)

review_parts = []
for review_path in REVIEW_PATHS:
    review_part = pd.read_parquet(review_path, columns=['user_id', 'date'])
    review_part['date'] = pd.to_datetime(review_part['date'])
    review_part['target_year'] = review_part['date'].dt.year
    review_part['target_month'] = review_part['date'].dt.month
    review_parts.append(review_part)

review_df = pd.concat(review_parts, ignore_index=True)
target_activity_df = (
    review_df.groupby(['user_id', 'target_year'], observed=True)
    .agg(
        derived_target_review_count=('target_month', 'size'),
        target_active_months=('target_month', 'nunique'),
    )
    .reset_index()
)

label_df = cohort_df.merge(
    target_activity_df,
    on=['user_id', 'target_year'],
    how='left',
    validate='many_to_one',
)
label_df[['derived_target_review_count', 'target_active_months']] = (
    label_df[['derived_target_review_count', 'target_active_months']].fillna(0).astype(int)
)
assert label_df['target_review_count'].astype(int).equals(label_df['derived_target_review_count'])

label_df['retention_state'] = np.select(
    [
        label_df['target_review_count'].eq(state_config['stopped_review_count']),
        label_df['target_review_count'].lt(state_config['retained_min_review_count'])
        | label_df['target_active_months'].lt(state_config['retained_min_active_months']),
    ],
    [state_config['stopped_class'], state_config['weakened_class']],
    default=state_config['retained_class'],
).astype('int8')
label_df['retention_state_label'] = label_df['retention_state'].map(CLASS_LABELS_KO)
assert label_df['churn'].eq(label_df['retention_state'].eq(2).astype('int8')).all()

distribution_df = (
    label_df.groupby(['selection_year', 'target_year', 'retention_state', 'retention_state_label'], observed=True)
    .size()
    .rename('users')
    .reset_index()
)
distribution_df['year_total'] = distribution_df.groupby('selection_year')['users'].transform('sum')
distribution_df['share'] = distribution_df['users'] / distribution_df['year_total']
distribution_df.to_csv(DISTRIBUTION_PATH, index=False, encoding='utf-8-sig')

label_columns = [
    'sample_id', 'target_review_count', 'target_active_months',
    'retention_state', 'retention_state_label',
]
frame = modeling_df.merge(label_df[label_columns], on='sample_id', how='inner', validate='one_to_one')
assert len(frame) == len(modeling_df) == 21601
assert not ({'churn', 'target_review_count', 'target_active_months', 'retention_state'} & set(feature_columns))
assert not frame[feature_columns].columns.duplicated().any()

def build_model() -> Pipeline:
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            C=0.1,
            penalty='l1',
            solver='saga',
            class_weight='balanced',
            max_iter=2500,
            tol=1e-3,
            random_state=SEED,
        )),
    ])

def threshold_predictions(scores: np.ndarray) -> np.ndarray:
    predictions = np.zeros(len(scores), dtype='int8')
    predictions[scores[:, 1] >= WEAKENED_THRESHOLD] = 1
    predictions[scores[:, 2] >= STOPPED_THRESHOLD] = 2
    return predictions

def evaluate(y_true: np.ndarray, predictions: np.ndarray, scores: np.ndarray) -> dict[str, float]:
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, predictions, labels=CLASS_CODES, zero_division=0
    )
    y_binary = label_binarize(y_true, classes=CLASS_CODES)
    record = {
        'accuracy': accuracy_score(y_true, predictions),
        'balanced_accuracy': balanced_accuracy_score(y_true, predictions),
        'macro_precision': float(precision.mean()),
        'macro_recall': float(recall.mean()),
        'macro_f1': f1_score(y_true, predictions, average='macro'),
        'weighted_f1': f1_score(y_true, predictions, average='weighted'),
        'macro_pr_auc': float(np.mean([
            average_precision_score(y_binary[:, i], scores[:, i]) for i in CLASS_CODES
        ])),
        'macro_ovr_roc_auc': roc_auc_score(y_true, scores, multi_class='ovr', average='macro'),
    }
    for index, class_name in enumerate(CLASS_NAMES):
        record[f'{class_name}_precision'] = precision[index]
        record[f'{class_name}_recall'] = recall[index]
        record[f'{class_name}_f1'] = f1[index]
        record[f'{class_name}_support'] = int(support[index])
        record[f'{class_name}_pr_auc'] = average_precision_score(y_binary[:, index], scores[:, index])
        record[f'{class_name}_roc_auc'] = roc_auc_score(y_binary[:, index], scores[:, index])
    return record

def confusion_records(split: str, policy: str, y_true: np.ndarray, predictions: np.ndarray) -> list[dict]:
    matrix = confusion_matrix(y_true, predictions, labels=CLASS_CODES)
    return [
        {
            'split': split,
            'decision_policy': policy,
            'actual_state': CLASS_NAMES[actual],
            'predicted_state': CLASS_NAMES[predicted],
            'users': int(matrix[actual, predicted]),
        }
        for actual in CLASS_CODES
        for predicted in CLASS_CODES
    ]

def top_k_records(split: str, y_true: np.ndarray, scores: np.ndarray) -> list[dict]:
    rankings = {
        'unified': scores[:, 1] + scores[:, 2],
        'stopped_only': scores[:, 2],
        'weakened_only': scores[:, 1],
    }
    status_loss = y_true != 0
    stopped = y_true == 2
    weakened = y_true == 1
    records = []
    for ranking_name, ranking_score in rankings.items():
        order = np.argsort(-ranking_score, kind='stable')
        for rate in TOP_K_RATES:
            users = int(np.ceil(len(y_true) * rate))
            selected = order[:users]
            captured_status = int(status_loss[selected].sum())
            captured_stopped = int(stopped[selected].sum())
            captured_weakened = int(weakened[selected].sum())
            precision = captured_status / users
            records.append({
                'split': split,
                'ranking': ranking_name,
                'target_rate': rate,
                'target_users': users,
                'status_loss_captured': captured_status,
                'status_loss_precision': precision,
                'status_loss_recall': captured_status / status_loss.sum(),
                'status_loss_lift': precision / status_loss.mean(),
                'stopped_captured': captured_stopped,
                'stopped_recall': captured_stopped / stopped.sum(),
                'weakened_captured': captured_weakened,
                'weakened_recall': captured_weakened / weakened.sum(),
            })
    return records

validation_records = []
confusion_rows = []
oof_parts = []
for fold, (train_end, validation_year) in enumerate(TIME_FOLDS, start=1):
    train_fold = frame[frame['selection_year'].le(train_end)]
    validation_fold = frame[frame['selection_year'].eq(validation_year)]
    fold_model = build_model()
    fold_model.fit(train_fold[feature_columns], train_fold['retention_state'])
    fold_scores = fold_model.predict_proba(validation_fold[feature_columns])
    fold_predictions = threshold_predictions(fold_scores)
    fold_record = {
        'record_type': 'fold',
        'split': f'fold_{fold}',
        'train_selection_years': f'2009~{train_end}',
        'validation_selection_year': validation_year,
        'train_samples': len(train_fold),
        'validation_samples': len(validation_fold),
        **evaluate(validation_fold['retention_state'].to_numpy(), fold_predictions, fold_scores),
    }
    validation_records.append(fold_record)
    confusion_rows.extend(confusion_records(
        f'validation_{validation_year}', 'threshold',
        validation_fold['retention_state'].to_numpy(), fold_predictions,
    ))
    oof_part = validation_fold[['sample_id', 'retention_state']].copy()
    oof_part['retained_score'] = fold_scores[:, 0]
    oof_part['weakened_score'] = fold_scores[:, 1]
    oof_part['stopped_score'] = fold_scores[:, 2]
    oof_parts.append(oof_part)

fold_result_df = pd.DataFrame(validation_records)
metric_columns = [column for column in fold_result_df.columns if column not in {
    'record_type', 'split', 'train_selection_years', 'validation_selection_year',
    'train_samples', 'validation_samples',
}]
summary_records = []
for statistic in ['mean', 'std']:
    summary = {
        'record_type': statistic,
        'split': 'time_5_fold',
        'train_selection_years': 'expanding_2009~2015',
        'validation_selection_year': pd.NA,
        'train_samples': pd.NA,
        'validation_samples': pd.NA,
    }
    for column in metric_columns:
        summary[column] = getattr(fold_result_df[column], statistic)(ddof=1) if statistic == 'std' else fold_result_df[column].mean()
    summary_records.append(summary)

oof_df = pd.concat(oof_parts, ignore_index=True)
oof_scores = oof_df[['retained_score', 'weakened_score', 'stopped_score']].to_numpy()
oof_y = oof_df['retention_state'].to_numpy()
oof_predictions = threshold_predictions(oof_scores)
oof_record = {
    'record_type': 'pooled_oof',
    'split': 'time_5_fold',
    'train_selection_years': 'expanding_2009~2015',
    'validation_selection_year': pd.NA,
    'train_samples': pd.NA,
    'validation_samples': len(oof_df),
    **evaluate(oof_y, oof_predictions, oof_scores),
}

final_train_df = frame[frame['selection_year'].le(2016)].copy()
final_test_df = frame[frame['selection_year'].eq(2017)].copy()
assert final_train_df['selection_year'].max() < final_test_df['selection_year'].min()
assert final_test_df['target_year'].eq(2019).all()

final_model = build_model()
final_model.fit(final_train_df[feature_columns], final_train_df['retention_state'])
test_scores = final_model.predict_proba(final_test_df[feature_columns])
test_predictions = threshold_predictions(test_scores)
test_y = final_test_df['retention_state'].to_numpy()
test_metrics = evaluate(test_y, test_predictions, test_scores)
test_record = {
    'record_type': 'final_test',
    'split': 'selection_2017_target_2019',
    'train_selection_years': '2009~2016',
    'validation_selection_year': 2017,
    'train_samples': len(final_train_df),
    'validation_samples': len(final_test_df),
    **test_metrics,
}

validation_result_df = pd.concat([
    fold_result_df,
    pd.DataFrame(summary_records),
    pd.DataFrame([oof_record, test_record]),
], ignore_index=True)
validation_result_df.to_csv(VALIDATION_PATH, index=False, encoding='utf-8-sig')

confusion_rows.extend(confusion_records('pooled_oof', 'threshold', oof_y, oof_predictions))
confusion_rows.extend(confusion_records('final_test', 'threshold', test_y, test_predictions))
confusion_df = pd.DataFrame(confusion_rows)
confusion_df.to_csv(CONFUSION_PATH, index=False, encoding='utf-8-sig')

top_k_df = pd.DataFrame(
    top_k_records('pooled_oof', oof_y, oof_scores)
    + top_k_records('final_test', test_y, test_scores)
)
top_k_df.to_csv(TOP_K_PATH, index=False, encoding='utf-8-sig')

profile_df = final_test_df.copy()
profile_df['retained_score'] = test_scores[:, 0]
profile_df['weakened_score'] = test_scores[:, 1]
profile_df['stopped_score'] = test_scores[:, 2]
profile_df['priority_score'] = profile_df['weakened_score'] + profile_df['stopped_score']
profile_df['predicted_state'] = test_predictions
profile_df['predicted_state_label'] = profile_df['predicted_state'].map(CLASS_LABELS_KO)
profile_df['priority_rank'] = profile_df['priority_score'].rank(method='first', ascending=False).astype(int)
profile_df['priority_top_percent'] = profile_df['priority_rank'] / len(profile_df) * 100
target_users = int(np.ceil(len(profile_df) * PRIMARY_TARGET_RATE))
profile_df['selected_for_crm'] = profile_df['priority_rank'].le(target_users).astype('int8')
profile_df = profile_df.sort_values(['priority_rank', 'sample_id']).reset_index(drop=True)
profile_df.to_parquet(PROFILE_PATH, index=False)

joblib.dump(final_model, MODEL_PATH)
reloaded_model = joblib.load(MODEL_PATH)
reloaded_scores = reloaded_model.predict_proba(final_test_df[feature_columns])
assert np.allclose(test_scores, reloaded_scores, rtol=0, atol=1e-12)
model_checksum = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()

imputed_feature_names = final_model.named_steps['imputer'].get_feature_names_out(feature_columns)
coefficients = final_model.named_steps['model'].coef_
active_columns = int((np.abs(coefficients) > 1e-9).any(axis=0).sum())
top20 = top_k_df[
    top_k_df['split'].eq('final_test')
    & top_k_df['ranking'].eq('unified')
    & top_k_df['target_rate'].eq(PRIMARY_TARGET_RATE)
].iloc[0]

metadata = {
    'version': 'v03',
    'model_name': 'Core Multiclass Logistic L1',
    'model_type': 'LogisticRegression',
    'problem_type': 'multiclass_classification',
    'class_map': {str(code): name for code, name in zip(CLASS_CODES, CLASS_NAMES)},
    'class_labels_ko': {str(key): value for key, value in CLASS_LABELS_KO.items()},
    'label_definition': {
        'retained': 'target_review_count >= 10 and target_active_months >= 3',
        'weakened': 'target_review_count >= 1 and (target_review_count < 10 or target_active_months < 3)',
        'stopped': 'target_review_count == 0',
    },
    'model_parameters': {
        'penalty': 'l1', 'C': 0.1, 'solver': 'saga',
        'class_weight': 'balanced', 'max_iter': 2500,
        'tol': 0.001, 'random_state': SEED,
    },
    'decision_thresholds': {
        'stopped_score': STOPPED_THRESHOLD,
        'weakened_score': WEAKENED_THRESHOLD,
        'evaluation_order': ['stopped', 'weakened', 'retained'],
    },
    'priority_policy': {
        'score': 'weakened_score + stopped_score',
        'primary_target_rate': PRIMARY_TARGET_RATE,
    },
    'feature_set': 'activity+interval+business',
    'feature_count': len(feature_columns),
    'feature_columns': feature_columns,
    'imputed_feature_count': len(imputed_feature_names),
    'active_imputed_feature_count': active_columns,
    'time_folds': [
        {'train_selection_years': f'2009~{train_end}', 'validation_selection_year': validation_year}
        for train_end, validation_year in TIME_FOLDS
    ],
    'final_train_selection_years': '2009~2016',
    'test_selection_year': 2017,
    'test_target_year': 2019,
    'test_metrics': {key: float(value) for key, value in test_metrics.items()},
    'model_sha256': model_checksum,
    'python_version': sys.version.split()[0],
    'sklearn_version': sklearn.__version__,
    'pandas_version': pd.__version__,
    'score_warning': '클래스별 점수는 확률 보정 전 모델 점수이며 실제 상태 확률로 표현하지 않는다.',
}
METADATA_PATH.write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

report = f'''# 파워 리뷰어 3클래스 리텐션 상태 모델 v03

- 상태: 프로토타입 기준 모델
- 모델: LogisticRegression, L1, C=0.1, class_weight=balanced
- 피처: 기존 Core 43개
- 최종 Train: 선정연도 2009~2016
- 최종 Test: 선정연도 2017, 타깃연도 2019

## 라벨

- 유지: 리뷰 10건 이상이고 활동 월 3개월 이상
- 약화: 리뷰 1건 이상이며 리뷰 10건 미만이거나 활동 월 3개월 미만
- 중단: 리뷰 0건

기존 churn은 중단 상태와 동일한 보조 검증 라벨로 보존한다.

## 최종 Test

| 지표 | 결과 |
|---|---:|
| Accuracy | {test_metrics['accuracy']:.2%} |
| Balanced Accuracy | {test_metrics['balanced_accuracy']:.2%} |
| Macro F1 | {test_metrics['macro_f1']:.4f} |
| Macro PR-AUC | {test_metrics['macro_pr_auc']:.4f} |
| Macro OvR ROC-AUC | {test_metrics['macro_ovr_roc_auc']:.4f} |

| 클래스 | Precision | Recall | F1 | PR-AUC |
|---|---:|---:|---:|---:|
| 유지 | {test_metrics['retained_precision']:.2%} | {test_metrics['retained_recall']:.2%} | {test_metrics['retained_f1']:.4f} | {test_metrics['retained_pr_auc']:.4f} |
| 약화 | {test_metrics['weakened_precision']:.2%} | {test_metrics['weakened_recall']:.2%} | {test_metrics['weakened_f1']:.4f} | {test_metrics['weakened_pr_auc']:.4f} |
| 중단 | {test_metrics['stopped_precision']:.2%} | {test_metrics['stopped_recall']:.2%} | {test_metrics['stopped_f1']:.4f} | {test_metrics['stopped_pr_auc']:.4f} |

## 통합 상위 20% 정책

- 관리 대상: {int(top20['target_users']):,}명
- 실제 지위 상실 포착: {int(top20['status_loss_captured']):,}명
- 지위 상실 Precision: {top20['status_loss_precision']:.2%}
- 지위 상실 Recall: {top20['status_loss_recall']:.2%}
- 지위 상실 Lift: {top20['status_loss_lift']:.2f}배
- 중단 포착: {int(top20['stopped_captured']):,}명, Recall {top20['stopped_recall']:.2%}
- 약화 포착: {int(top20['weakened_captured']):,}명, Recall {top20['weakened_recall']:.2%}

지위 상실은 Test의 약 63%이므로 Top 20% 정책의 Recall 상한이 낮다.
통합 큐는 자동 상태 확정이 아니라 관리자의 우선 검토 순위로 사용한다.

## 제한사항

- 약화 사용자의 Top 20% 순위 개선은 중단 사용자보다 작다.
- 클래스별 모델 점수는 확률 보정 전이다.
- CRM 개입 효과와 복귀 결과는 현재 데이터에 없다.
- 선정연도 2017 Test 결과를 확인한 후 모델·규제·임계값을 추가 조정하지 않는다.
'''
REPORT_PATH.write_text(report, encoding='utf-8')

expected_paths = [
    MODEL_PATH, METADATA_PATH, PROFILE_PATH, DISTRIBUTION_PATH,
    VALIDATION_PATH, CONFUSION_PATH, TOP_K_PATH, REPORT_PATH,
]
assert all(path.exists() and path.stat().st_size > 0 for path in expected_paths)
assert len(profile_df) == 4157
assert int(profile_df['selected_for_crm'].sum()) == 832

print('v03 artifacts created')
print('final_test_samples:', len(profile_df))
print('macro_pr_auc:', round(test_metrics['macro_pr_auc'], 6))
print('macro_f1:', round(test_metrics['macro_f1'], 6))
print('active_imputed_features:', active_columns, '/', len(imputed_feature_names))
print('model_sha256:', model_checksum)
display(validation_result_df.tail(4).round(4))
display(top_k_df[(top_k_df['split'] == 'final_test') & (top_k_df['ranking'] == 'unified')].round(4))


v03 artifacts created
final_test_samples: 4157
macro_pr_auc: 0.598582
macro_f1: 0.575408
active_imputed_features: 33 / 49
model_sha256: c9b2bf574796686dd8996fd2c2d1991cbab5ea94573e282860d3a9fc908333f1


,record_type,split,train_selection_years,validation_selection_year,train_samples,validation_samples,accuracy,balanced_accuracy,macro_precision,macro_recall,...,weakened_f1,weakened_support,weakened_pr_auc,weakened_roc_auc,stopped_precision,stopped_recall,stopped_f1,stopped_support,stopped_pr_auc,stopped_roc_auc
5,mean,time_5_fold,expanding_2009~2015,<NA>,<NA>,<NA>,0.5829,0.5932,0.5649,0.5932,...,0.5534,1336.0000,0.6225,0.6962,0.3579,0.6142,0.4517,436.6000,0.3803,0.7988
6,std,time_5_fold,expanding_2009~2015,<NA>,<NA>,<NA>,0.0171,0.0105,0.0088,0.0105,...,0.0195,416.9328,0.0098,0.0127,0.0173,0.0290,0.0127,117.0419,0.0121,0.0192
7,pooled_oof,time_5_fold,expanding_2009~2015,<NA>,<NA>,14312,0.5806,0.5928,0.5641,0.5928,...,0.5557,6680.0000,0.6240,0.6955,0.3538,0.6175,0.4499,2183.0000,0.3758,0.7962
8,final_test,selection_2017_target_2019,2009~2016,2017,17444,4157,0.5915,0.5943,0.5713,0.5943,...,0.5749,1948.0000,0.6130,0.6903,0.3920,0.5851,0.4695,670.0000,0.4166,0.8147


,split,ranking,target_rate,target_users,status_loss_captured,status_loss_precision,status_loss_recall,status_loss_lift,stopped_captured,stopped_recall,weakened_captured,weakened_recall
24,final_test,unified,0.05,208,203,0.9760,0.0775,1.5497,106,0.1582,97,0.0498
25,final_test,unified,0.10,416,394,0.9471,0.1505,1.5039,190,0.2836,204,0.1047
26,final_test,unified,0.15,624,584,0.9359,0.2231,1.4861,265,0.3955,319,0.1638
27,final_test,unified,0.20,832,773,0.9291,0.2953,1.4753,329,0.4910,444,0.2279
28,final_test,unified,0.25,1040,958,0.9212,0.3659,1.4627,388,0.5791,570,0.2926
29,final_test,unified,0.30,1248,1137,0.9111,0.4343,1.4466,449,0.6701,688,0.3532
30,final_test,unified,0.35,1455,1310,0.9003,0.5004,1.4296,500,0.7463,810,0.4158
31,final_test,unified,0.40,1663,1486,0.8936,0.5676,1.4189,545,0.8134,941,0.4831


## v03 피처 중요도

위 셀에서 학습된 `final_model`, `final_train_df`, `final_test_df`를 그대로 재사용한다.

- 개별 피처 중요도: 이미 학습된 `final_model`에 대한 Permutation Importance만 계산한다. 재학습 없음.
- 그룹 중요도: 활동량/작성 간격/탐색 그룹을 하나씩 제거하고 같은 하이퍼파라미터로 다시 학습해 macro PR-AUC 하락폭을 비교한다(v02 `final_feature_group_importance_v02.csv`와 동일 방법론). 이 재학습은 설명용 임시 모델이며 `MODEL_PATH`에 저장된 운영 모델을 덮어쓰지 않는다.


In [ ]:
from sklearn.inspection import permutation_importance

FEATURE_IMPORTANCE_PATH = REPORT_TABLE_DIR / 'final_feature_importance_v03.csv'
FEATURE_GROUP_IMPORTANCE_PATH = REPORT_TABLE_DIR / 'final_feature_group_importance_v03.csv'

FEATURE_GROUPS = {
    'baseline_review_count': 'activity', 'baseline_active_months': 'activity',
    'baseline_reviews_per_active_month': 'activity', 'recent_review_count': 'activity',
    'recent_active_months': 'activity', 'recent_reviews_per_active_month': 'activity',
    'review_count_diff': 'activity', 'review_count_ratio': 'activity',
    'review_count_decline_rate': 'activity', 'active_month_diff': 'activity',
    'active_month_ratio': 'activity', 'active_month_decline_rate': 'activity',
    'reviews_per_active_month_diff': 'activity', 'reviews_per_active_month_ratio': 'activity',
    'reviews_per_active_month_decline_rate': 'activity',
    'baseline_mean_interval_days': 'interval', 'baseline_median_interval_days': 'interval',
    'baseline_max_interval_days': 'interval', 'baseline_recency_days': 'interval',
    'recent_mean_interval_days': 'interval', 'recent_median_interval_days': 'interval',
    'recent_max_interval_days': 'interval', 'recent_recency_days': 'interval',
    'recent_interval_available': 'interval', 'mean_interval_increase_days': 'interval',
    'median_interval_increase_days': 'interval', 'max_interval_increase_days': 'interval',
    'recency_increase_days': 'interval',
    'baseline_unique_business_count': 'business', 'recent_unique_business_count': 'business',
    'recent_revisited_business_count': 'business', 'recent_new_vs_baseline_count': 'business',
    'unique_business_count_diff': 'business', 'unique_business_ratio': 'business',
    'unique_business_decline_rate': 'business', 'recent_revisit_rate': 'business',
    'recent_new_vs_baseline_rate': 'business', 'baseline_new_business_count': 'business',
    'recent_new_business_count': 'business', 'baseline_new_business_rate': 'business',
    'recent_new_business_rate': 'business', 'new_business_count_diff': 'business',
    'new_business_rate_decline': 'business',
}
FEATURE_GROUP_LABELS_KO = {'activity': '리뷰 활동량', 'interval': '작성 간격', 'business': '음식점 탐색'}
assert set(FEATURE_GROUPS) == set(feature_columns)

def macro_pr_auc_scorer(estimator, X, y) -> float:
    proba = estimator.predict_proba(X)
    y_binary = label_binarize(y, classes=CLASS_CODES)
    return float(np.mean([average_precision_score(y_binary[:, i], proba[:, i]) for i in CLASS_CODES]))

# 1) 개별 피처 중요도 — 재학습 없음, 이미 학습된 final_model만 사용
perm = permutation_importance(
    final_model, final_test_df[feature_columns], test_y,
    scoring=macro_pr_auc_scorer, n_repeats=20, random_state=SEED, n_jobs=-1,
)
feature_importance_df = pd.DataFrame({
    'feature': feature_columns,
    'feature_group': [FEATURE_GROUPS[f] for f in feature_columns],
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=False).reset_index(drop=True)
positive_sum = feature_importance_df['importance_mean'].clip(lower=0).sum()
feature_importance_df['importance_share_pct'] = (
    feature_importance_df['importance_mean'].clip(lower=0) / positive_sum * 100
    if positive_sum > 0 else 0.0
)
feature_importance_df['feature_group_label'] = feature_importance_df['feature_group'].map(FEATURE_GROUP_LABELS_KO)
feature_importance_df.insert(0, 'rank', np.arange(1, len(feature_importance_df) + 1))
feature_importance_df.to_csv(FEATURE_IMPORTANCE_PATH, index=False, encoding='utf-8-sig')

# 2) 그룹 중요도 — 그룹 하나씩 제거 후 임시 재학습(설명 전용, MODEL_PATH는 덮어쓰지 않음)
baseline_pr_auc = test_metrics['macro_pr_auc']
test_y_binary = label_binarize(test_y, classes=CLASS_CODES)
group_rows = []
for group in ['activity', 'interval', 'business']:
    reduced_columns = [f for f in feature_columns if FEATURE_GROUPS[f] != group]
    ablated_model = build_model()
    ablated_model.fit(final_train_df[reduced_columns], final_train_df['retention_state'])
    ablated_scores = ablated_model.predict_proba(final_test_df[reduced_columns])
    ablated_pr_auc = float(np.mean([
        average_precision_score(test_y_binary[:, i], ablated_scores[:, i]) for i in CLASS_CODES
    ]))
    group_rows.append({
        'feature_group': group,
        'feature_count': sum(1 for f in feature_columns if FEATURE_GROUPS[f] == group),
        'importance_mean': baseline_pr_auc - ablated_pr_auc,
        'importance_std': np.nan,
        'baseline_pr_auc': baseline_pr_auc,
        'feature_group_label': FEATURE_GROUP_LABELS_KO[group],
    })

group_importance_df = pd.DataFrame(group_rows).sort_values('importance_mean', ascending=False)
group_importance_df.insert(0, 'rank', np.arange(1, len(group_importance_df) + 1))
group_importance_df = group_importance_df[
    ['feature_group', 'feature_count', 'importance_mean', 'importance_std',
     'baseline_pr_auc', 'rank', 'feature_group_label']
]
group_importance_df.to_csv(FEATURE_GROUP_IMPORTANCE_PATH, index=False, encoding='utf-8-sig')

print('saved:', FEATURE_IMPORTANCE_PATH)
print('saved:', FEATURE_GROUP_IMPORTANCE_PATH)
display(feature_importance_df.head(10))
display(group_importance_df)
